# Figure 4: Model fits to cancer incidence data across cancer subtypes

This notebook reproduces Figure 4 of the manuscript.

The notebook uses:

- `TCR_size_diversity_age_sex.csv` or the Zenodo-hosted public TCR table
- `export_filtered_073125.csv`
- `final_parameters_5.csv` if needed for checking fitted parameters

The file `export_filtered_073125.csv` contains the subtype-specific SEER incidence data exported for the manuscript analysis.

The main figure shows, for each cancer subtype:

- male incidence data,
- female incidence data,
- model fits using sex-specific TCR diversity curves,
- uncertainty bands obtained by propagating uncertainty in the TCR diversity curves.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.interpolate import CubicSpline, interp1d

%config InlineBackend.figure_format = "pdf"

plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)

In [2]:
# ============================================================
# Configuration
# ============================================================

TOFF = 5

# These filenames follow the original notebook.
# Place them in the same directory as this notebook.
INCIDENCE_FILE = "export_filtered_073125.csv"
PARAMETER_FILE = "final_parameters_5.csv"

# The TCR table can either be local or read directly from Zenodo.
TCR_FILE = "TCR_size_diversity_age_sex.csv"
TCR_URL = "https://zenodo.org/records/13993996/files/TCR_size_diversity_age_sex.csv?download=1"

# Figure output.
SAVE_FIGURE = False
FIGURE_FILE = "fig4.pdf"

# Figure layout.
FIGURE_SIZE = (18, 8)
N_COLS = 6

# Figure 4 uses the original fit and TCR-diversity uncertainty bands.
# Bootstrap is only needed if regenerating confidence intervals for parameters,
# so keep this small here unless you want to rerun the full parameter bootstrap.
N_BOOTSTRAP = 11

FILTER = [
    "hodgkin_lymphoma",
    "colorectal_carcinoma_msi_high_medullary_subtype",
]

In [3]:
# ============================================================
# Load TCR repertoire data
# ============================================================
#
# Use local TCR file if present; otherwise fall back to Zenodo.

if os.path.exists(TCR_FILE):
    tcr_path = TCR_FILE
else:
    tcr_path = TCR_URL

df_features_pd = (
    pd.read_csv(tcr_path)
    .rename(
        columns={
            "#Subject_ID": "Subject_ID",
            " age": "age",
            " sex": "sex",
            " size": "ntemplates",
            " diversity": "nur",
        }
    )
)

df_features_pd["sex"] = df_features_pd["sex"].str.strip().str.lower()

df_features_pd.head()

,Subject_ID,age,sex,ntemplates,nur
0,30000,63,male,508139,274815
1,30001,76,female,263716,167611
2,30002,70,female,519416,261836
3,30003,44,male,576294,383460
4,30004,47,male,505447,290642


In [4]:
# ============================================================
# Age bins and original binned-median helper
# ============================================================

b1 = np.arange(9) * 10
b2 = np.arange(9) * 10 + 10
b2[-1] = 100

bins = [(lo, hi) for lo, hi in zip(b1, b2)]


def bootbin_median(
    _x,
    _y,
    nbins=20,
    nreal=1001,
    ci=[0.1, 0.9],
    bins=None,
    do_mean=False,
    do_sum=False,
    plot_bootbin=False,
):
    """
    Original manuscript helper for binned medians.

    Parameters
    ----------
    _x, _y : array-like
        Input data to bin.
    nbins : int
        Number of equal-count bins if explicit bins are not provided.
    nreal : int
        Number of bootstrap realizations used to estimate uncertainty
        on the binned median.
    ci : list
        Quantiles used for shaded confidence intervals.
    bins : list of tuple or None
        Explicit bin edges. If provided, bins are interpreted as
        [lower, upper) intervals.
    do_mean, do_sum : bool
        Alternative aggregation modes retained from the original notebook.
    plot_bootbin : bool
        If True, plot the binned summary.

    Returns
    -------
    dict
        Dictionary containing xmed, ymed, yerr, ci_lo, ci_hi, and yres.
    """
    good_index = np.where(np.isfinite(_x) & np.isfinite(_y))[0]

    x = np.asarray(_x)[good_index]
    y = np.asarray(_y)[good_index]

    ss = np.argsort(x)
    xss = x[ss]
    yss = y[ss]

    if bins is None:
        x_split = np.array_split(xss, nbins)
        y_split = np.array_split(yss, nbins)
    else:
        x_split = []
        y_split = []

        for bi in bins:
            in_bin = np.where((xss >= bi[0]) & (xss < bi[1]))[0]

            if len(in_bin) > 1:
                x_split.append(xss[in_bin])
                y_split.append(yss[in_bin])

    if do_mean:
        agg_func_x = np.mean
        agg_func_y = np.mean
    elif do_sum:
        agg_func_x = np.median
        agg_func_y = np.sum
    else:
        agg_func_x = np.median
        agg_func_y = np.median

    xmed = np.asarray([agg_func_x(xi) for xi in x_split])
    ymed = np.asarray([agg_func_y(yi) for yi in y_split])

    yres = np.asarray(
        [
            yj
            for yi in y_split
            for yj in yi - agg_func_y(yi)
        ]
    )

    ci_lo = np.asarray(
        [
            interp1d(
                np.linspace(0, 1, num=len(yi)),
                np.sort(yi),
            )(ci[0])
            for yi in y_split
        ]
    )

    ci_hi = np.asarray(
        [
            interp1d(
                np.linspace(0, 1, num=len(yi)),
                np.sort(yi),
            )(ci[1])
            for yi in y_split
        ]
    )

    if not do_sum:
        yboot = [
            [
                agg_func_y(
                    yi[np.random.randint(0, high=len(yi), size=len(yi))]
                )
                for yi in y_split
            ]
            for _ in range(nreal)
        ]

        yerr = np.asarray(
            [
                np.nanstd(yi)
                for yi in np.asarray(yboot).T
            ]
        )
    else:
        yerr = np.sqrt(ymed)

    if plot_bootbin:
        plt.figure(figsize=(10, 7))
        plt.fill_between(xmed, ci_lo, ci_hi, alpha=0.2)
        plt.errorbar(xmed, ymed, yerr=yerr)

    return {
        "xmed": xmed,
        "ymed": ymed,
        "yres": yres,
        "yerr": yerr,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
    }

In [5]:
# ============================================================
# Load subtype-specific SEER incidence data
# ============================================================

data = pd.read_csv(INCIDENCE_FILE)

# The queried data sometimes has values such as "~" instead of NaN.
data = data.replace("~", 0)

# Keep all sites present in the file.
sites_to_keep = data["Site"].unique().tolist()

print(f"Original data: {len(data):,} rows")

data = data[data["Site"].isin(sites_to_keep)]

print(f"Filtered data: {len(data):,} rows")

# Convert numeric columns from string to numeric.
numeric_cols = [
    "Count",
    "Population",
    "Incidence",
    "SE",
    "LowerCI",
    "UpperCI",
]

for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(
            data[col].astype(str).str.replace(",", ""),
            errors="coerce",
        )

data.head()

Original data: 1,260 rows
Filtered data: 1,260 rows


,Site,Age,Sex,Incidence,SE,LowerCI,UpperCI,Count,Population
0,Lung adenocarcinoma,00 years,Male and female,0.014,0.006,0.005,0.030,6,43061201
1,Lung adenocarcinoma,00 years,Male,0.005,0.005,0.000,0.025,1,22011981
2,Lung adenocarcinoma,00 years,Female,0.024,0.011,0.008,0.055,5,21049220
3,Lung adenocarcinoma,01-04 years,Male and female,0.000,0.000,0.000,0.002,0,172923682
4,Lung adenocarcinoma,01-04 years,Male,0.000,0.000,0.000,0.004,0,88392817


In [6]:
# ============================================================
# Convert incidence data into site_sex_data dictionary
# ============================================================
#
# The original notebook stores each site/sex combination as:
# [Age, Incidence, SE, LowerCI, UpperCI, Count, Population]

grouped_data = data[data["Sex"].isin(["Male", "Female"])]
unique_combinations = grouped_data[["Site", "Sex"]].drop_duplicates()

site_sex_data = {}
sites_to_keep_cleaned = []


def clean_site_name(site):
    """
    Match the site-name cleaning convention used in the original notebook.
    """
    return (
        site.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace(",", "")
        .replace("-", "_")
        .replace("/", "_")
    )


for _, row in unique_combinations.iterrows():
    site = row["Site"]
    sex = row["Sex"]

    filtered_data = grouped_data[
        (grouped_data["Site"] == site)
        & (grouped_data["Sex"] == sex)
    ].sort_values("Age")

    result_list = filtered_data[
        [
            "Age",
            "Incidence",
            "SE",
            "LowerCI",
            "UpperCI",
            "Count",
            "Population",
        ]
    ].values.tolist()

    site_name_cleaned = clean_site_name(site)
    var_name = f"{site_name_cleaned}_{sex.lower()}"

    site_sex_data[var_name] = result_list

    if site_name_cleaned not in sites_to_keep_cleaned:
        sites_to_keep_cleaned.append(site_name_cleaned)

print(f"Total site/sex entries: {len(site_sex_data)}")
print(f"Total cleaned sites: {len(sites_to_keep_cleaned)}")

first_key = list(site_sex_data.keys())[0]
print(first_key)
site_sex_data[first_key][:2]

Total site/sex entries: 40
Total cleaned sites: 20
lung_adenocarcinoma_male


[['00 years', 0.005, 0.005, 0.0, 0.025, 1, 22011981],
 ['01-04 years', 0.0, 0.0, 0.0, 0.004, 0, 88392817]]

In [7]:
# ============================================================
# Trim age ranges
# ============================================================
#
# The original notebook trims to the age range used for model fitting.
# This removes very young bins and the oldest open-ended bins.

for var_name in site_sex_data:
    site_sex_data[var_name] = site_sex_data[var_name][5:-3]

first_key = list(site_sex_data.keys())[0]

print(f"Example key: {first_key}")
print(f"N age bins after trimming: {len(site_sex_data[first_key])}")
print("First three rows:")
site_sex_data[first_key][:3]

Example key: lung_adenocarcinoma_male
N age bins after trimming: 13
First three rows:


[['20-24 years', 0.064, 0.007, 0.05, 0.08, 75, 117600125],
 ['25-29 years', 0.15, 0.011, 0.129, 0.174, 176, 117171061],
 ['30-34 years', 0.471, 0.02, 0.432, 0.512, 546, 115938128]]

In [8]:
# ============================================================
# Model fitting and TCR-bootstrap functions for Figure 4
# ============================================================
#
# This cell contains the code that fits the cancer incidence model
# for each cancer subtype.
#
# The fit uses:
# - female and male SEER incidence curves for a given cancer subtype
# - sex-specific TCR diversity curves inferred from repertoire data
# - one shared parameter set for both sexes
#
# Sex is not included as an explicit model variable. Sex differences enter
# only through the female and male TCR diversity curves.
#
# Model:
#
# I(t, D) = A * (t - toff)^gamma * exp(-D(t - toff) / R0)
#           / (1 + exp[-k(t - t0)])
#
# In the code below, R0 corresponds to D0 in the manuscript.

# ------------------------------------------------------------
# Age parsing helpers
# ------------------------------------------------------------

def keep_age_range(row):
    """
    Return True if the age group is usable.

    SEER exports sometimes contain 'Unknown' age groups, which are
    excluded from model fitting.
    """
    return row[0] != "Unknown"


def get_age_midpoint(age_str):
    """
    Convert a SEER age-bin string to its midpoint.

    Examples
    --------
    '20-24 years' -> 22
    '80-84 years' -> 82
    '85+ years'   -> 87
    """
    if "years" not in age_str:
        return 0

    age_label = age_str.split(" ")[0]

    if "-" in age_label:
        start, end = age_label.split("-")
        return (int(start) + int(end)) / 2

    if "+" in age_label:
        return int(age_label.split("+")[0]) + 2

    return int(age_label)


def get_age_minimum(age_str):
    """
    Convert a SEER age-bin string to its lower edge.

    Used only for axis tick labels in Figure 4.
    """
    if "years" not in age_str:
        return 0

    age_label = age_str.split(" ")[0]

    if "-" in age_label:
        return int(age_label.split("-")[0])

    if "+" in age_label:
        return int(age_label.split("+")[0])

    return int(age_label)


# ------------------------------------------------------------
# Cancer-incidence extraction
# ------------------------------------------------------------

def extract_cancer_data(site_name, site_sex_data):
    """
    Extract age, incidence, and standard error arrays for one cancer subtype.

    Parameters
    ----------
    site_name : str
        Cleaned cancer subtype name.
    site_sex_data : dict
        Dictionary with keys like '<site>_female' and '<site>_male'.

    Returns
    -------
    ages : np.ndarray
        Age-bin midpoints.
    age_labels : list
        Original SEER age-bin labels.
    frate, frate_err : np.ndarray
        Female incidence rates and standard errors.
    mrate, mrate_err : np.ndarray
        Male incidence rates and standard errors.
    """
    female_data = site_sex_data[f"{site_name}_female"]
    male_data = site_sex_data[f"{site_name}_male"]

    female_data = [row for row in female_data if keep_age_range(row)]
    male_data = [row for row in male_data if keep_age_range(row)]

    ages = np.asarray([get_age_midpoint(row[0]) for row in female_data])
    age_labels = [row[0] for row in female_data]

    frate = np.asarray([row[1] for row in female_data], dtype=float)
    frate_err = np.asarray([row[2] for row in female_data], dtype=float)

    mrate = np.asarray([row[1] for row in male_data], dtype=float)
    mrate_err = np.asarray([row[2] for row in male_data], dtype=float)

    return ages, age_labels, frate, frate_err, mrate, mrate_err


# ------------------------------------------------------------
# TCR diversity processing
# ------------------------------------------------------------

def process_tcr_data(df_features_pd, bins):
    """
    Build sex-specific TCR diversity curves.

    The model uses TCR diversity in linear units, not log units.
    We estimate median diversity as a function of age separately for
    females and males, then interpolate those curves with cubic splines.

    Returns
    -------
    fx, mx : CubicSpline
        Female and male median TCR diversity curves.
    fe, me : CubicSpline
        Bootstrap uncertainty on the female and male curves.
    mind, find : np.ndarray
        Male and female subject indices used for bootstrap resampling.
    """
    mind = np.where(df_features_pd["sex"].values == "male")[0]
    find = np.where(df_features_pd["sex"].values == "female")[0]

    ci1 = [0.16, 0.84]

    male_diversity = bootbin_median(
        df_features_pd["age"].values[mind],
        df_features_pd["nur"].values[mind],
        bins=bins,
        ci=ci1,
    )

    female_diversity = bootbin_median(
        df_features_pd["age"].values[find],
        df_features_pd["nur"].values[find],
        bins=bins,
        ci=ci1,
    )

    fx = CubicSpline(
        female_diversity["xmed"],
        female_diversity["ymed"],
    )

    mx = CubicSpline(
        male_diversity["xmed"],
        male_diversity["ymed"],
    )

    fe = CubicSpline(
        female_diversity["xmed"],
        female_diversity["yerr"],
    )

    me = CubicSpline(
        male_diversity["xmed"],
        male_diversity["yerr"],
    )

    return fx, mx, fe, me, mind, find


# ------------------------------------------------------------
# Model functions
# ------------------------------------------------------------

def create_fitting_function(fx, mx, toff=5):
    """
    Create the cancer-incidence model for a fixed latency offset.

    The age array passed to this function is a concatenation of:
    [female ages, male ages].

    The first el entries use the female TCR diversity curve.
    The remaining entries use the male TCR diversity curve.
    """
    def model(age, A, R0, gamma, k, t0, el):
        female_age = age[:el]
        male_age = age[el:]

        D = np.concatenate(
            [
                fx(female_age - toff),
                mx(male_age - toff),
            ]
        )

        immune_component = np.exp(-D / R0)
        mutation_component = (age - toff) ** gamma
        logistic_component = 1.0 / (1.0 + np.exp(-k * (age - t0)))

        return A * immune_component * mutation_component * logistic_component

    return model


def create_error_functions(fx, mx, fe, me, toff=5):
    """
    Create upper and lower model curves by propagating uncertainty in
    the TCR diversity curves.

    The manuscript uses ±2 sigma uncertainty in TCR diversity to generate
    the shaded confidence bands.
    """
    def model_upper(age, A, R0, gamma, k, t0, el):
        female_age = age[:el]
        male_age = age[el:]

        D = np.concatenate(
            [
                fx(female_age - toff) + 2 * fe(female_age - toff),
                mx(male_age - toff) + 2 * me(male_age - toff),
            ]
        )

        immune_component = np.exp(-D / R0)
        mutation_component = (age - toff) ** gamma
        logistic_component = 1.0 / (1.0 + np.exp(-k * (age - t0)))

        return A * immune_component * mutation_component * logistic_component

    def model_lower(age, A, R0, gamma, k, t0, el):
        female_age = age[:el]
        male_age = age[el:]

        D = np.concatenate(
            [
                fx(female_age - toff) - 2 * fe(female_age - toff),
                mx(male_age - toff) - 2 * me(male_age - toff),
            ]
        )

        immune_component = np.exp(-D / R0)
        mutation_component = (age - toff) ** gamma
        logistic_component = 1.0 / (1.0 + np.exp(-k * (age - t0)))

        return A * immune_component * mutation_component * logistic_component

    return model_upper, model_lower


# ------------------------------------------------------------
# Fit original SEER incidence data
# ------------------------------------------------------------

def fit_original_data(model, ages, frate, frate_err, mrate, mrate_err):
    """
    Fit the cancer-incidence model to one cancer subtype.

    The same parameter set is fit jointly to female and male incidence.
    The female and male predictions differ only because the model uses
    sex-specific TCR diversity curves.

    Parameters
    ----------
    model : callable
        Model returned by create_fitting_function().
    ages : np.ndarray
        Age-bin midpoints.
    frate, mrate : np.ndarray
        Female and male incidence rates.
    frate_err, mrate_err : np.ndarray
        Standard errors of female and male incidence.

    Returns
    -------
    par_orig : np.ndarray
        Best-fit parameters [A, R0, gamma, k, t0].
    """
    el_fit = len(ages)

    def model_wrapper(age, A, R0, gamma, k, t0):
        return model(age, A, R0, gamma, k, t0, el_fit)

    # Bounds from the original manuscript notebook.
    parameter_bounds = (
        [0, 1e3, 0, -1.0, 20.0],
        [1e5, 1e8, 10, 0.0, 120.0],
    )

    initial_guess = [1, 1e5, 2, -0.5, 75.0]

    par_orig, _ = curve_fit(
        model_wrapper,
        np.concatenate([ages, ages]),
        np.concatenate([frate, mrate]),
        sigma=np.concatenate([frate_err, mrate_err]),
        p0=initial_guess,
        bounds=parameter_bounds,
        maxfev=100000,
    )

    return par_orig


# ------------------------------------------------------------
# Bootstrap TCR diversity curves and refit
# ------------------------------------------------------------

def run_bootstrap_iterations(
    df_features_pd,
    mind,
    find,
    bins,
    ages,
    frate,
    frate_err,
    mrate,
    mrate_err,
    par_orig,
    n_bootstrap=101,
    toff=5,
):
    """
    Propagate TCR diversity uncertainty into fitted parameters.

    Each bootstrap realization:
    - resamples male and female TCR repertoires independently,
    - recomputes sex-specific median TCR diversity curves,
    - refits the cancer-incidence model to the original SEER incidence data.

    Cancer incidence data are not bootstrapped here. The dominant uncertainty
    propagated in this procedure is uncertainty in the TCR diversity curves.
    """
    parameter_bounds = (
        [0, 1e3, 0, -1.0, 20.0],
        [1e5, 1e8, 10, 0.0, 120.0],
    )

    bootstrapped_params = []
    el_fit = len(ages)
    ci1 = [0.16, 0.84]

    for i in range(n_bootstrap):
        try:
            mrand = np.random.choice(
                np.arange(len(mind)),
                size=len(mind),
                replace=True,
            )

            frand = np.random.choice(
                np.arange(len(find)),
                size=len(find),
                replace=True,
            )

            male_diversity_boot = bootbin_median(
                df_features_pd["age"].values[mind][mrand],
                df_features_pd["nur"].values[mind][mrand],
                bins=bins,
                ci=ci1,
                nreal=11,
            )

            female_diversity_boot = bootbin_median(
                df_features_pd["age"].values[find][frand],
                df_features_pd["nur"].values[find][frand],
                bins=bins,
                ci=ci1,
                nreal=11,
            )

            fx_boot = CubicSpline(
                female_diversity_boot["xmed"],
                female_diversity_boot["ymed"],
            )

            mx_boot = CubicSpline(
                male_diversity_boot["xmed"],
                male_diversity_boot["ymed"],
            )

            def model_boot(age, A, R0, gamma, k, t0):
                female_age = age[:el_fit]
                male_age = age[el_fit:]

                D = np.concatenate(
                    [
                        fx_boot(female_age - toff),
                        mx_boot(male_age - toff),
                    ]
                )

                immune_component = np.exp(-D / R0)
                mutation_component = (age - toff) ** gamma
                logistic_component = 1.0 / (1.0 + np.exp(-k * (age - t0)))

                return (
                    A
                    * immune_component
                    * mutation_component
                    * logistic_component
                )

            par_boot, _ = curve_fit(
                model_boot,
                np.concatenate([ages, ages]),
                np.concatenate([frate, mrate]),
                sigma=np.concatenate([frate_err, mrate_err]),
                p0=par_orig,
                bounds=parameter_bounds,
                maxfev=10000,
            )

            bootstrapped_params.append(par_boot)

        except Exception as error:
            print(f"Bootstrap iteration {i} failed: {error}")

    return np.asarray(bootstrapped_params)


def calculate_parameter_stats(bootstrapped_params, par_orig):
    """
    Calculate bootstrap confidence intervals using the original-centered
    convention used in the manuscript.

    The reported central parameter values are the fits to the original data.
    Bootstrap realizations define the 95% confidence intervals.
    """
    if len(bootstrapped_params) == 0:
        param_ci_lower = par_orig.copy()
        param_ci_upper = par_orig.copy()
        param_stds = np.zeros_like(par_orig)
        param_means = par_orig.copy()
        return param_means, param_stds, param_ci_lower, param_ci_upper

    param_ci_lower = np.percentile(bootstrapped_params, 2.5, axis=0)
    param_ci_upper = np.percentile(bootstrapped_params, 97.5, axis=0)

    param_means = par_orig.copy()
    param_stds = np.std(bootstrapped_params, axis=0)

    return param_means, param_stds, param_ci_lower, param_ci_upper


# ------------------------------------------------------------
# Generate fitted curves for plotting
# ------------------------------------------------------------

def generate_fitted_curves(model, model_upper, model_lower, parameters, ages):
    """
    Generate smooth female and male model curves for plotting.
    """
    age_int = np.linspace(ages[0], ages[-1], 100)
    el = len(age_int)

    model_values = model(
        np.concatenate([age_int, age_int]),
        *parameters,
        el,
    )

    fitf = model_values[:el]
    fitm = model_values[el:]

    upper_values = model_upper(
        np.concatenate([age_int, age_int]),
        *parameters,
        el,
    )

    fitf_u = upper_values[:el]
    fitm_u = upper_values[el:]

    lower_values = model_lower(
        np.concatenate([age_int, age_int]),
        *parameters,
        el,
    )

    fitf_l = lower_values[:el]
    fitm_l = lower_values[el:]

    return age_int, fitf, fitm, fitf_u, fitm_u, fitf_l, fitm_l


# ------------------------------------------------------------
# Main fitting function for one cancer subtype
# ------------------------------------------------------------

def bootstrap_cancer_site_simple(
    site_name,
    df_features_pd,
    bins,
    site_sex_data,
    n_bootstrap=300,
    create_individual_plot_flag=False,
    toff=5,
):
    """
    Fit one cancer subtype and optionally bootstrap TCR diversity curves.

    This is the central fitting routine used to generate Figure 4.
    """
    try:
        ages, age_labels, frate, frate_err, mrate, mrate_err = extract_cancer_data(
            site_name,
            site_sex_data,
        )

        fx, mx, fe, me, mind, find = process_tcr_data(
            df_features_pd,
            bins,
        )

        model = create_fitting_function(fx, mx, toff=toff)
        model_upper, model_lower = create_error_functions(
            fx,
            mx,
            fe,
            me,
            toff=toff,
        )

        par_orig = fit_original_data(
            model,
            ages,
            frate,
            frate_err,
            mrate,
            mrate_err,
        )

        print(
            f"  Original fit: "
            f"A={par_orig[0]:.2e}, "
            f"R0={par_orig[1]:.2e}, "
            f"gamma={par_orig[2]:.2f}, "
            f"k={par_orig[3]:.3f}, "
            f"t0={par_orig[4]:.1f}"
        )

        age_int, fitf, fitm, fitf_u, fitm_u, fitf_l, fitm_l = generate_fitted_curves(
            model,
            model_upper,
            model_lower,
            par_orig,
            ages,
        )

        bootstrapped_params = None
        bootstrap_success = False

        param_means = par_orig.copy()
        param_stds = np.zeros_like(par_orig)
        param_ci_lower = par_orig.copy()
        param_ci_upper = par_orig.copy()

        if n_bootstrap > 0:
            print(f"  Running {n_bootstrap} TCR bootstrap iterations...")

            bootstrapped_params = run_bootstrap_iterations(
                df_features_pd,
                mind,
                find,
                bins,
                ages,
                frate,
                frate_err,
                mrate,
                mrate_err,
                par_orig,
                n_bootstrap=n_bootstrap,
                toff=toff,
            )

            if len(bootstrapped_params) >= 10:
                (
                    param_means,
                    param_stds,
                    param_ci_lower,
                    param_ci_upper,
                ) = calculate_parameter_stats(
                    bootstrapped_params,
                    par_orig,
                )

                bootstrap_success = True

                print(
                    f"  ✓ TCR bootstrap successful: "
                    f"{len(bootstrapped_params)}/{n_bootstrap} fits"
                )
            else:
                print(
                    f"  ⚠ TCR bootstrap insufficient: "
                    f"{len(bootstrapped_params)}/{n_bootstrap} fits"
                )

        return {
            "site": site_name,
            "ages": ages,
            "age_labels": age_labels,
            "frate": frate,
            "frate_err": frate_err,
            "mrate": mrate,
            "mrate_err": mrate_err,
            "age_int": age_int,
            "fitf": fitf,
            "fitm": fitm,
            "fitf_u": fitf_u,
            "fitm_u": fitm_u,
            "fitf_l": fitf_l,
            "fitm_l": fitm_l,
            "A_orig": par_orig[0],
            "R0_orig": par_orig[1],
            "gamma_orig": par_orig[2],
            "k_orig": par_orig[3],
            "t0_orig": par_orig[4],
            "A_mean": param_means[0],
            "A_std": param_stds[0],
            "A_ci_lower": param_ci_lower[0],
            "A_ci_upper": param_ci_upper[0],
            "R0_mean": param_means[1],
            "R0_std": param_stds[1],
            "R0_ci_lower": param_ci_lower[1],
            "R0_ci_upper": param_ci_upper[1],
            "gamma_mean": param_means[2],
            "gamma_std": param_stds[2],
            "gamma_ci_lower": param_ci_lower[2],
            "gamma_ci_upper": param_ci_upper[2],
            "k_mean": param_means[3],
            "k_std": param_stds[3],
            "k_ci_lower": param_ci_lower[3],
            "k_ci_upper": param_ci_upper[3],
            "t0_mean": param_means[4],
            "t0_std": param_stds[4],
            "t0_ci_lower": param_ci_lower[4],
            "t0_ci_upper": param_ci_upper[4],
            "par_mean": param_means,
            "bootstrapped_params": bootstrapped_params,
            "n_bootstrap_success": (
                len(bootstrapped_params)
                if bootstrapped_params is not None
                else 0
            ),
            "bootstrap_success": bootstrap_success,
            "n_age_groups": len(ages),
            "fit_success": True,
        }

    except Exception as error:
        print(f"  ✗ {site_name} failed: {error}")
        return None

In [9]:
# ============================================================
# Fit all Figure 4 cancer subtypes
# ============================================================

# Ordered as in the manuscript Figure 4.
site_order = [
    "acute_myeloid_leukemia_aml",
    "chronic_myeloid_leukemia_cml",
    "chronic_lymphocytic_leukemia_cll",
    "diffuse_large_b_cell_lymphoma_dlbcl",
    "follicular_lymphoma",
    "multiple_myeloma",
    "colorectal_adenocarcinoma_mss",
    "gastric_adenocarcinoma",
    "pancreatic_adenocarcinoma",
    "lung_adenocarcinoma",
    "lung_squamous_cell_carcinoma",
    "small_cell_lung_carcinoma",
    "kidney_chromophobe_carcinoma",
    "kidney_clear_cell_carcinoma",
    "kidney_papillary_carcinoma",
    "malignant_gliomas_astrocytic_tumors",
    "melanoma_of_skin",
    "soft_tissue_sarcomas",
]


site_labels = {
    "acute_myeloid_leukemia_aml": "Acute Myeloid\nLeukemia",
    "chronic_myeloid_leukemia_cml": "Chronic Myeloid\nLeukemia",
    "chronic_lymphocytic_leukemia_cll": "Chronic Lymphocytic\nLeukemia",
    "diffuse_large_b_cell_lymphoma_dlbcl": "Diffuse Large B-cell\nLymphoma",
    "follicular_lymphoma": "Follicular\nLymphoma",
    "multiple_myeloma": "Multiple\nMyeloma",
    "colorectal_adenocarcinoma_mss": "Colorectal\nAdenocarcinoma MSS",
    "gastric_adenocarcinoma": "Gastric\nAdenocarcinoma",
    "pancreatic_adenocarcinoma": "Pancreatic\nAdenocarcinoma",
    "lung_adenocarcinoma": "Lung\nAdenocarcinoma",
    "lung_squamous_cell_carcinoma": "Lung Squamous Cell\nCarcinoma",
    "small_cell_lung_carcinoma": "Small Cell Lung\nCarcinoma",
    "kidney_chromophobe_carcinoma": "Kidney Chromophobe\nCarcinoma",
    "kidney_clear_cell_carcinoma": "Kidney Clear Cell\nCarcinoma",
    "kidney_papillary_carcinoma": "Kidney Papillary\nCarcinoma",
    "malignant_gliomas_astrocytic_tumors": "Malignant Gliomas\n(Astrocytic Tumors)",
    "melanoma_of_skin": "Melanoma\nof Skin",
    "soft_tissue_sarcomas": "Soft Tissue\nSarcomas",
}

# For the public figure notebook, keep bootstrap modest by default.
# Increase to the manuscript setting if desired.
N_BOOTSTRAP_FIG4 = 11

figure4_results = {}

for site in site_order:
    if f"{site}_female" not in site_sex_data or f"{site}_male" not in site_sex_data:
        print(f"Skipping {site}: missing male or female incidence data")
        continue

    print(f"\nFitting {site}...")

    figure4_results[site] = bootstrap_cancer_site_simple(
        site_name=site,
        df_features_pd=df_features_pd,
        bins=bins,
        site_sex_data=site_sex_data,
        n_bootstrap=N_BOOTSTRAP_FIG4,
        create_individual_plot_flag=False,
        toff=TOFF,
    )

print("\nFinished fitting Figure 4 subtypes.")
print(f"N successful fits: {sum(v is not None for v in figure4_results.values())}")


Fitting acute_myeloid_leukemia_aml...
  Original fit: A=5.58e-01, R0=8.85e+04, gamma=1.45, k=-1.000, t0=84.2
  Running 11 TCR bootstrap iterations...
  ✓ TCR bootstrap successful: 11/11 fits

Fitting chronic_myeloid_leukemia_cml...
  Original fit: A=1.09e+01, R0=9.82e+04, gamma=0.42, k=-0.249, t0=87.6
  Running 11 TCR bootstrap iterations...
  ✓ TCR bootstrap successful: 11/11 fits

Fitting chronic_lymphocytic_leukemia_cll...
  Original fit: A=7.60e-07, R0=7.30e+04, gamma=5.28, k=-0.129, t0=66.0
  Running 11 TCR bootstrap iterations...
  ✓ TCR bootstrap successful: 11/11 fits

Fitting diffuse_large_b_cell_lymphoma_dlbcl...
  Original fit: A=1.31e-01, R0=1.04e+05, gamma=1.88, k=-0.563, t0=84.5
  Running 11 TCR bootstrap iterations...
  ✓ TCR bootstrap successful: 11/11 fits

Fitting follicular_lymphoma...
  Original fit: A=1.84e-05, R0=2.91e+05, gamma=3.57, k=-0.092, t0=75.5
  Running 11 TCR bootstrap iterations...
  ✓ TCR bootstrap successful: 11/11 fits

Fitting multiple_myeloma...
 

In [10]:
# ============================================================
# Optional: regenerate final_parameters_0/5/10.csv
#           and bootstrap_parameters_toff*_*.csv
# ============================================================
#
# This reproduces the original notebook convention.
#
# For each TOFF value, this writes:
#
#   final_parameters_{TOFF}.csv
#       Compact best-fit table used downstream.
#
#   bootstrap_parameters_toff*_*.csv
#       Summary table containing original best-fit parameters and
#       asymmetric bootstrap-derived errors.
#
# The bootstrap_parameters files are NOT raw bootstrap draws.

RUN_TOFF_SWEEP = False
TOFF_VALUES = [0, 5, 10]

# Use 0 for fast regeneration of final_parameters only.
# Use a positive value to regenerate bootstrap-derived uncertainty summaries.
N_BOOTSTRAP_FOR_SWEEP = 101 # Set to 10001 to match paper

EXPORT_FINAL_PARAMETERS = False
EXPORT_BOOTSTRAP_SUMMARIES = False

BOOTSTRAP_FILENAME_MAP = {
    0: "bootstrap_parameters_toff0_082025.csv",
    5: "bootstrap_parameters_toff5_082125.csv",
    10: "bootstrap_parameters_toff10_082025.csv",
}

if RUN_TOFF_SWEEP:
    for toff_value in TOFF_VALUES:
        print(f"\n{'=' * 70}")
        print(f"Running fits for TOFF = {toff_value}")
        print(f"{'=' * 70}")

        sweep_results = {}

        for site in site_order:
            if (
                f"{site}_female" not in site_sex_data
                or f"{site}_male" not in site_sex_data
            ):
                print(f"Skipping {site}: missing male or female incidence data")
                continue

            print(f"\nFitting {site}...")

            sweep_results[site] = bootstrap_cancer_site_simple(
                site_name=site,
                df_features_pd=df_features_pd,
                bins=bins,
                site_sex_data=site_sex_data,
                n_bootstrap=N_BOOTSTRAP_FOR_SWEEP,
                create_individual_plot_flag=False,
                toff=toff_value,
            )

        # ----------------------------------------------------
        # Export compact final_parameters_{TOFF}.csv
        # ----------------------------------------------------
        if EXPORT_FINAL_PARAMETERS:
            final_rows = []

            for site in site_order:
                result = sweep_results.get(site)

                if result is None:
                    continue

                final_rows.append(
                    {
                        "#site": site,
                        "A": result["A_orig"],
                        "D0": result["R0_orig"],
                        "gamma": result["gamma_orig"],
                        "k": result["k_orig"],
                        "t0": result["t0_orig"],
                    }
                )

            final_table = pd.DataFrame(final_rows)

            final_file = f"final_parameters_{toff_value}.csv"
            final_table.to_csv(final_file, index=False)

            print(f"\nSaved {final_file}")

        # ----------------------------------------------------
        # Export original-style bootstrap summary CSV
        # ----------------------------------------------------
        if EXPORT_BOOTSTRAP_SUMMARIES:
            summary_rows = []

            for site in site_order:
                result = sweep_results.get(site)

                if result is None:
                    continue

                summary_rows.append(
                    {
                        "site": site,

                        "A_orig": result["A_orig"],
                        "A_lower": result["A_orig"] - result["A_ci_lower"],
                        "A_upper": result["A_ci_upper"] - result["A_orig"],

                        "R0_orig": result["R0_orig"],
                        "R0_lower": result["R0_orig"] - result["R0_ci_lower"],
                        "R0_upper": result["R0_ci_upper"] - result["R0_orig"],

                        "gamma_orig": result["gamma_orig"],
                        "gamma_lower": result["gamma_orig"] - result["gamma_ci_lower"],
                        "gamma_upper": result["gamma_ci_upper"] - result["gamma_orig"],

                        "k_orig": result["k_orig"],
                        "k_lower": result["k_orig"] - result["k_ci_lower"],
                        "k_upper": result["k_ci_upper"] - result["k_orig"],

                        "t0_orig": result["t0_orig"],
                        "t0_lower": result["t0_orig"] - result["t0_ci_lower"],
                        "t0_upper": result["t0_ci_upper"] - result["t0_orig"],
                    }
                )

            bootstrap_summary = pd.DataFrame(summary_rows)

            bootstrap_file = BOOTSTRAP_FILENAME_MAP.get(
                toff_value,
                f"bootstrap_parameters_toff{toff_value}.csv",
            )

            bootstrap_summary.to_csv(bootstrap_file, index=False)

            print(f"Saved {bootstrap_file}")
            display(bootstrap_summary.head())

In [11]:
# ============================================================
# Plot Figure 4 using original manuscript plotting logic
# ============================================================

def create_faceted_cancer_incidence_plots_original(
    results,
    ncols=6,
    figsize=(18, 8),
    save=False,
    plot_name="fig4.pdf",
):
    """
    Original-style Figure 4 plotting function.

    This intentionally follows the original manuscript plotting logic:
    - linear y-axis
    - one panel per cancer subtype
    - data plotted at age-bin midpoints
    - x-axis tick labels shown only on bottom row
    - y-axis label shown only on left column
    """
    desired_order = [
        "acute_myeloid_leukemia_aml",
        "chronic_lymphocytic_leukemia_cll",
        "chronic_myeloid_leukemia_cml",
        "diffuse_large_b_cell_lymphoma_dlbcl",
        "follicular_lymphoma",
        "multiple_myeloma",
        "lung_adenocarcinoma",
        "lung_squamous_cell_carcinoma",
        "small_cell_lung_carcinoma",
        "kidney_clear_cell_carcinoma",
        "kidney_papillary_carcinoma",
        "kidney_chromophobe_carcinoma",
        "colorectal_adenocarcinoma_mss",
        "gastric_adenocarcinoma",
        "pancreatic_adenocarcinoma",
        "melanoma_of_skin",
        "malignant_gliomas_astrocytic_tumors",
        "soft_tissue_sarcomas",
    ]

    custom_titles = [
        "Acute myeloid leukemia",
        "Chronic lymphocytic leukemia",
        "Chronic myeloid leukemia",
        "Diffuse large B cell lymphoma",
        "Follicular lymphoma",
        "Multiple myeloma",
        "Lung adenocarcinoma",
        "Lung squamous cell carcinoma",
        "Small cell lung carcinoma",
        "Kidney clear cell carcinoma",
        "Kidney papillary carcinoma",
        "Kidney chromophobe carcinoma",
        "Colorectal adenocarcinoma",
        "Gastric adenocarcinoma",
        "Pancreatic adenocarcinoma",
        "Melanoma of skin",
        "Malignant gliomas",
        "Soft tissue sarcomas",
    ]

    successful_results = [
        r for r in results
        if r is not None and r.get("fit_success", False)
    ]

    if len(successful_results) == 0:
        print("No successful fits to plot.")
        return None, None

    def get_sort_key(result):
        site = result["site"]
        if site in desired_order:
            return desired_order.index(site)
        return len(desired_order)

    successful_results.sort(key=get_sort_key)

    nrows = int(np.ceil(len(successful_results) / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
    )

    if nrows == 1:
        axes = axes.reshape(1, -1)

    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for idx, result in enumerate(successful_results):
        row = idx // ncols
        col = idx % ncols
        ax = axes[row, col]

        ages = result["ages"]
        age_labels = result["age_labels"]

        frate = result["frate"]
        frate_err = result["frate_err"]
        mrate = result["mrate"]
        mrate_err = result["mrate_err"]

        age_int = result["age_int"]
        fitf = result["fitf"]
        fitm = result["fitm"]
        fitf_u = result["fitf_u"]
        fitm_u = result["fitm_u"]
        fitf_l = result["fitf_l"]
        fitm_l = result["fitm_l"]

        # Uncertainty bands.
        ax.fill_between(age_int, fitm_l, fitm_u, alpha=0.2, color=colors[0])
        ax.fill_between(age_int, fitf_l, fitf_u, alpha=0.2, color=colors[1])

        # Incidence data.
        ax.errorbar(
            ages,
            mrate,
            yerr=mrate_err,
            color=colors[0],
            marker="o",
            markersize=6,
            linestyle="",
            capsize=3,
            alpha=0.7,
        )

        ax.errorbar(
            ages,
            frate,
            yerr=frate_err,
            color=colors[1],
            marker="v",
            markersize=6,
            linestyle="",
            capsize=3,
            alpha=0.7,
        )

        # Fitted curves.
        ax.plot(age_int, fitm, color=colors[0], linestyle="-", linewidth=2, alpha=0.8)
        ax.plot(age_int, fitf, color=colors[1], linestyle="-", linewidth=2, alpha=0.8)

        title = custom_titles[idx] if idx < len(custom_titles) else result["site"].replace("_", " ").title()
        ax.set_title(title, fontsize=12, fontweight="bold")

        ax.grid(False)

        # Original x-axis behavior: ticks at age-bin midpoints,
        # labels show lower edge of age bin.
        min_ages = [get_age_minimum(label) for label in age_labels]
        ax.set_xticks(ages)

        if row == nrows - 1:
            ax.set_xticklabels([f"{min_age}" for min_age in min_ages], fontsize=9)
            ax.set_xlabel("Age", fontsize=10)
        else:
            ax.set_xticklabels([])

        if col == 0:
            ax.set_ylabel(
                "Cancer incidence\n[per 100,000 person-years]",
                fontsize=10,
            )

        ax.tick_params(axis="y", labelsize=9)

    # Remove unused axes.
    total_subplots = nrows * ncols
    for idx in range(len(successful_results), total_subplots):
        row = idx // ncols
        col = idx % ncols
        axes[row, col].remove()

    plt.tight_layout(pad=0.05)
    plt.subplots_adjust(
        top=0.95,
        bottom=0.1,
        hspace=0.15,
        wspace=0.175,
    )

    if save:
        plt.savefig(plot_name, dpi=300, bbox_inches="tight")
        print(f"Figure saved: {plot_name}")

    return fig, axes


# The original plotting function expects a list of result dictionaries,
# not a dictionary keyed by site.
figure4_results_list = [
    result for result in figure4_results.values()
    if result is not None
]

fig, axes = create_faceted_cancer_incidence_plots_original(
    figure4_results_list,
    ncols=N_COLS,
    figsize=FIGURE_SIZE,
    save=SAVE_FIGURE,
    plot_name=FIGURE_FILE,
)

plt.show()

<Figure size 1800x800 with 18 Axes>